# 02 — Titanic Predictive Modeling

This notebook continues from the same committed `titanic.csv` produced immediately after the single raw-data load in `01_eda.ipynb`. No second `sns.load_dataset()` call is made here.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

ROOT = Path.cwd()
if ROOT.name != "analytics":
    ROOT = ROOT / "analytics"

df = pd.read_csv(ROOT / "titanic.csv")
print(df.shape)
display(df.head())

## 1. Modeling frame and stratified split

The classification target is `survived`. We remove identifiers and columns that are not appropriate predictive inputs for this exercise. The split is stratified so the survivor/non-survivor proportions remain comparable between train and test.

In [ ]:
model_df = df.copy()

features = [
    "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"
]
X = model_df[features].copy()
y = model_df["survived"].astype(int)

print("Class balance:")
display(y.value_counts(normalize=True).rename("proportion"))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion"))
print("Test target proportions:")
display(y_test.value_counts(normalize=True).rename("proportion"))

## 2. Train-only preprocessing

The `ColumnTransformer` is inside each model pipeline. Therefore its imputer, encoder and scaler are fitted only when the pipeline is fitted on `X_train`; test data is passed through `transform` internally without refitting.

In [ ]:
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

## 3. Train three classifiers on the identical split

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
}

fitted = {}
rows = []
roc_data = {}

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe

    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "auc": roc_auc_score(y_test, prob),
    })
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_data[name] = (fpr, tpr)

metrics_df = pd.DataFrame(rows).set_index("model")
display(metrics_df)

In [ ]:
# Confusion matrices.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pipe) in zip(axes, fitted.items()):
    pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

# ROC comparison.
fig, ax = plt.subplots(figsize=(8, 6))
for name, (fpr, tpr) in roc_data.items():
    auc = metrics_df.loc[name, "auc"]
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves")
ax.legend()
plt.show()

## 4. Decision tree visualization

In [ ]:
tree_pipe = fitted["Decision Tree"]
feature_names = tree_pipe.named_steps["preprocessor"].get_feature_names_out()
tree_model = tree_pipe.named_steps["model"]

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=["not_survived", "survived"],
    filled=False,
    rounded=True,
    max_depth=3,
    fontsize=8,
    ax=ax,
)
ax.set_title("Decision Tree (first three levels)")
plt.show()

## 5. Imbalance comparison

The same train/test split is retained. SMOTE is applied only to the training fold, never to the test fold.

In [ ]:
print("Overall class balance:")
display(y.value_counts().rename_axis("survived").to_frame("count"))

def evaluate_pipe(pipe):
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    return {
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }

base_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])
balanced_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])
smote_pipe = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

imbalance_results = pd.DataFrame({
    "baseline": evaluate_pipe(base_pipe),
    "class_weight_balanced": evaluate_pipe(balanced_pipe),
    "SMOTE_train_only": evaluate_pipe(smote_pipe),
}).T

display(imbalance_results)

best_strategy = imbalance_results["f1"].idxmax()
print(
    "Observed imbalance-strategy result:",
    best_strategy,
    "has the highest F1 on this fixed test split. "
    "Precision and recall should also be considered because changing class balance can trade one against the other."
)

## 6. Random Forest GridSearchCV and OOB score

The grid search chooses the best hyperparameters using cross-validation. The selected parameters are then used to construct a separate `RandomForestClassifier(oob_score=True, ...)` so that the requested OOB score is available.

In [ ]:
rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_base),
])

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10],
    "model__max_features": ["sqrt", "log2"],
}

grid = GridSearchCV(
    rf_pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    refit=True,
)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

best_rf_params = {
    key.replace("model__", ""): value
    for key, value in grid.best_params_.items()
}

rf_oob_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        **best_rf_params,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )),
])
rf_oob_pipeline.fit(X_train, y_train)
print("OOB score:", rf_oob_pipeline.named_steps["model"].oob_score_)

## 7. Fare regression side-task

Fare is the continuous target. It is excluded from the predictors to avoid target leakage. The same general train/test principle is used, with preprocessing fitted on the regression training split only.

In [ ]:
reg_features = ["survived", "pclass", "sex", "age", "sibsp", "parch", "embarked"]
Xr = model_df[reg_features].copy()
yr = model_df["fare"].astype(float)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.20, random_state=42
)

reg_num = ["survived", "pclass", "age", "sibsp", "parch"]
reg_cat = ["sex", "embarked"]

reg_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), reg_num),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), reg_cat),
])

reg_pipeline = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("model", LinearRegression()),
])
reg_pipeline.fit(Xr_train, yr_train)

reg_pred = reg_pipeline.predict(Xr_test)
mae = mean_absolute_error(yr_test, reg_pred)
rmse = mean_squared_error(yr_test, reg_pred) ** 0.5
r2 = r2_score(yr_test, reg_pred)
n = len(yr_test)
p = reg_pipeline.named_steps["preprocessor"].transform(Xr_test).shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

reg_metrics = pd.Series({
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "Adjusted_R2": adj_r2,
})
display(reg_metrics.to_frame("value"))

In [ ]:
residuals = yr_test - reg_pred
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(x=reg_pred, y=residuals, alpha=0.6, ax=ax)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Predicted fare")
ax.set_ylabel("Residual")
ax.set_title("Fare regression residual plot")
plt.show()

print(
    "Heteroscedasticity assessment: inspect whether residual spread changes "
    "systematically with fitted fare. A funnel/widening pattern would indicate "
    "non-constant variance; a roughly even cloud around zero would not."
)

**Regression interpretation.** The residual plot is the evidence for the heteroscedasticity conclusion. The metric values quantify prediction error and explained variance, while adjusted R² accounts for the number of encoded predictors. Classification and regression metrics are kept separate because their scales and meanings differ.

## 8. Final comparison tables

In [ ]:
classification_table = metrics_df[["accuracy", "precision", "recall", "f1", "auc"]].copy()
print("Classification metrics")
display(classification_table)

print("Regression metrics")
display(reg_metrics.to_frame("value"))

print("Combined presentation with separate metric groups:")
combined = classification_table.copy()
for metric in ["MAE", "RMSE", "R2", "Adjusted_R2"]:
    combined[metric] = np.nan
combined.loc["Linear Regression", ["MAE", "RMSE", "R2", "Adjusted_R2"]] = reg_metrics.values
display(combined)

### Deployment recommendation

Use the classifier whose observed test-set metrics best match the operational objective, with particular attention to recall and F1 if missing a true survivor is more costly than generating a false positive. The table above provides accuracy, precision, recall, F1 and AUC for the same held-out test set, so the final decision should cite the actual values rather than relying on model type alone. If two models are close, the simpler model and operational interpretability may also matter. The final fitted pipeline below is selected using the highest observed F1 among the three classifiers on this fixed test split.

In [ ]:
best_model_name = metrics_df["f1"].idxmax()
best_pipeline = fitted[best_model_name]

artifact_path = ROOT / "artifacts" / "best_pipeline.joblib"
artifact_path.parent.mkdir(exist_ok=True)
joblib.dump(best_pipeline, artifact_path)

print("Saved complete pipeline:", artifact_path)
print("Selected model:", best_model_name)

## 9. Reload the complete pipeline and predict raw input

The saved artifact contains both preprocessing and the estimator. No manually preprocessed input is needed.

In [ ]:
reloaded = joblib.load(artifact_path)

raw_example = X_test.head(3).copy()
predictions = reloaded.predict(raw_example)

display(raw_example.assign(predicted_survived=predictions))
print("Reloaded artifact predicts directly from raw feature columns.")